# Versioned polygenic-score calculation

M7 uses explicitly selected local GRCh38 additive models and exact canonical alleles. Missing markers stay missing; partial sums are not published complete scores. Raw scores are not probabilities, diagnoses, absolute risks, treatment advice, or portable percentiles. Personal contribution artifacts remain private. Colab is a Google-managed VM; use the local workflow if cloud processing is unacceptable.


In [ ]:
import os
import sys

in_colab = "google.colab" in sys.modules
PROFILE = os.environ.get(
    "GENOME_EVIDENCE_PROFILE", "personal_drive" if in_colab else "synthetic_ci"
)
REPOSITORY_URL = "https://github.com/jcollins-bioinfo/genome-evidence.git"
REPOSITORY_REF = os.environ.get("GENOME_EVIDENCE_GIT_REF", "main")
WORKSPACE_ROOT = os.environ.get(
    "GENOME_EVIDENCE_WORKSPACE", "/content/drive/MyDrive/genome-evidence-private"
)
SUBJECT_ID = os.environ.get("GENOME_EVIDENCE_SUBJECT_ID", "subject-0001")

In [ ]:
import importlib
import importlib.metadata
import json
import subprocess
from hashlib import sha256
from pathlib import Path

if PROFILE not in {"personal_drive", "synthetic_ci"}:
    raise ValueError("PROFILE must be personal_drive or synthetic_ci")

CHECKOUT = Path("/content/genome-evidence-src")
if PROFILE == "personal_drive":
    if "google.colab" in sys.modules:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
    if CHECKOUT.exists():
        remote = subprocess.run(
            ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
            timeout=30,
        ).stdout.strip()
        if remote != REPOSITORY_URL:
            raise RuntimeError("Unexpected checkout remote; move the checkout aside and rerun")
        dirty = subprocess.run(
            ["git", "-C", str(CHECKOUT), "status", "--porcelain"],
            check=True,
            capture_output=True,
            text=True,
            timeout=30,
        ).stdout
        if dirty:
            raise RuntimeError("Checkout is dirty; preserve or move it aside and rerun")
    else:
        subprocess.run(
            ["git", "clone", "--no-checkout", REPOSITORY_URL, str(CHECKOUT)],
            check=True,
            timeout=180,
        )
    subprocess.run(
        ["git", "-C", str(CHECKOUT), "fetch", "--force", "origin", REPOSITORY_REF],
        check=True,
        timeout=180,
    )
    RESOLVED_COMMIT = subprocess.run(
        ["git", "-C", str(CHECKOUT), "rev-parse", "--verify", "FETCH_HEAD^{commit}"],
        check=True,
        capture_output=True,
        text=True,
        timeout=30,
    ).stdout.strip()
    subprocess.run(
        ["git", "-C", str(CHECKOUT), "checkout", "--detach", RESOLVED_COMMIT],
        check=True,
        timeout=60,
    )
    previously_imported = sys.modules.get("genome_evidence")
    if previously_imported is not None:
        previous_file = Path(getattr(previously_imported, "__file__", "")).resolve()
        if not previous_file.is_relative_to(CHECKOUT.resolve()):
            raise RuntimeError(
                "genome_evidence was already imported elsewhere; restart the runtime"
            )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "-e",
            f"{CHECKOUT}[notebook]",
        ],
        check=True,
        timeout=600,
    )
    source_root = (CHECKOUT / "src").resolve()
    package_init = source_root / "genome_evidence" / "__init__.py"
    if not package_init.is_file():
        raise RuntimeError("Resolved checkout does not contain the genome_evidence package")
    source_path = str(source_root)
    if source_path not in sys.path:
        sys.path.insert(0, source_path)
    importlib.invalidate_caches()
else:
    RESOLVED_COMMIT = "installed-ci-package"

genome_evidence = importlib.import_module("genome_evidence")
PACKAGE_ORIGIN = Path(genome_evidence.__file__).resolve()
if PROFILE == "personal_drive" and not PACKAGE_ORIGIN.is_relative_to(CHECKOUT.resolve()):
    raise RuntimeError("genome_evidence import origin is outside the resolved checkout")
INSTALLED_VERSION = importlib.metadata.version("genome-evidence")
LOCK_SHA256 = (
    sha256((CHECKOUT / "uv.lock").read_bytes()).hexdigest() if PROFILE == "personal_drive" else None
)
SANITIZED_IMPORT_PATH = (
    str(PACKAGE_ORIGIN.relative_to(CHECKOUT))
    if PROFILE == "personal_drive"
    else "installed-ci-package"
)
BOOTSTRAP_STATUS = {
    "profile": PROFILE,
    "requested_ref": REPOSITORY_REF,
    "resolved_commit": RESOLVED_COMMIT,
    "version": INSTALLED_VERSION,
    "import_path": SANITIZED_IMPORT_PATH,
    "lock_sha256": LOCK_SHA256,
    "lock_equivalent": PROFILE != "personal_drive",
}
print(json.dumps(BOOTSTRAP_STATUS, sort_keys=True))

In [ ]:
from genome_evidence.workspace import validate_workspace

if PROFILE == "personal_drive":
    workspace = validate_workspace(Path(WORKSPACE_ROOT))
else:
    assert PROFILE == "synthetic_ci"
    workspace = None

In [ ]:
PGS_IDS = [value for value in os.environ.get("GENOME_EVIDENCE_PGS_IDS", "").split(",") if value]
SCORE_BUNDLE = os.environ.get("GENOME_EVIDENCE_SCORE_BUNDLE")
NORMALIZATION_RUN = os.environ.get("GENOME_EVIDENCE_NORMALIZATION_RUN")
GENOTYPE_SOURCE_POLICY = "observed_only"
IMPUTATION_RUN = None
REFERENCE_DISTRIBUTION = None

In [ ]:
import tempfile
from hashlib import sha256 as file_sha256

import polars as pl

from genome_evidence.ingest import Ingest23andMeConfig, ingest_23andme
from genome_evidence.normalization import NormalizationConfig, normalize_m1_run
from genome_evidence.polygenic_scoring import (
    ScoreConfig,
    calculate_polygenic_scores,
    validate_polygenic_score_bundle,
)

if PROFILE == "synthetic_ci":
    work = Path(tempfile.mkdtemp(prefix="genome-evidence-fabricated-m7-"))
    source = work / "fabricated-genotypes.txt"
    source.write_text(
        "# SYNTHETIC TEST DATA - NOT A REAL PERSON\n"
        "# source build: GRCh38\n"
        "# rsid\tchromosome\tposition\tgenotype\n"
        "rsFabricated1\t1\t101\tAG\n"
        "rsFabricated2\t1\t202\tCC\n"
    )
    markers = work / "fabricated-markers.json"
    markers.write_text(
        json.dumps(
            [
                {
                    "marker_id": "rsFabricated1",
                    "assembly": "GRCh38",
                    "chromosome": "1",
                    "position": 101,
                    "reference": "A",
                    "alternate": "G",
                    "orientation": "none",
                    "orientation_authoritative": True,
                },
                {
                    "marker_id": "rsFabricated2",
                    "assembly": "GRCh38",
                    "chromosome": "1",
                    "position": 202,
                    "reference": "C",
                    "alternate": "T",
                    "orientation": "none",
                    "orientation_authoritative": True,
                },
            ]
        )
    )
    fasta = work / "fabricated-grch38.fa"
    fasta.write_text(">1\n" + "A" * 101 + "A" * 100 + "C" + "A" * 20 + "\n")
    m1 = work / "m1"
    ingest_23andme(source, m1, Ingest23andMeConfig(genome_build_override="GRCh38"))
    m2 = work / "m2"
    normalize_m1_run(
        m1, m2, NormalizationConfig(marker_definitions=markers, target_reference=fasta)
    )
    bundle = work / "fabricated-score-bundle"
    bundle.mkdir()
    metadata = {
        "models": [
            {
                "pgs_id": "PGS999999",
                "version": "fabricated-v1",
                "trait": "fabricated trait",
                "source_url": "https://example.invalid/fabricated",
                "citation": "synthetic fixture",
                "license": "CC0 synthetic fixture",
                "assembly": "GRCh38",
                "declared_variant_count": 3,
                "completeness_policy": "all_supported_variants_required",
            }
        ]
    }
    (bundle / "model_metadata.json").write_text(json.dumps(metadata))
    pl.DataFrame(
        [
            {
                "pgs_id": "PGS999999",
                "assembly": "GRCh38",
                "chromosome": "1",
                "position": 101,
                "reference": "A",
                "alternate": "G",
                "effect_allele": "G",
                "effect_weight": "1.5",
            },
            {
                "pgs_id": "PGS999999",
                "assembly": "GRCh38",
                "chromosome": "1",
                "position": 202,
                "reference": "C",
                "alternate": "T",
                "effect_allele": "C",
                "effect_weight": "-2.5e-1",
            },
            {
                "pgs_id": "PGS999999",
                "assembly": "GRCh38",
                "chromosome": "2",
                "position": 10,
                "reference": "A",
                "alternate": "C",
                "effect_allele": "C",
                "effect_weight": "0.25",
            },
        ]
    ).write_parquet(bundle / "model_variants.parquet")
    pl.DataFrame(schema={"pgs_id": pl.String, "reason": pl.String}).write_parquet(
        bundle / "model_exclusions.parquet"
    )
    artifacts = {}
    for name in ("model_metadata.json", "model_variants.parquet", "model_exclusions.parquet"):
        artifact = bundle / name
        artifacts[name] = {
            "byte_size": artifact.stat().st_size,
            "sha256": file_sha256(artifact.read_bytes()).hexdigest(),
            "privacy_class": "public_or_controlled_aggregate",
            "redistribution": "synthetic_fixture",
        }
    (bundle / "manifest.json").write_text(
        json.dumps(
            {
                "schema": "genome-evidence-pgs-bundle/v1",
                "bundle_id": "fabricated-pgs-bundle-v1",
                "artifacts": artifacts,
            }
        )
    )
    selected_ids = ("PGS999999",)
    output = work / "m7"
else:
    if not PGS_IDS:
        raise ValueError("PGS_IDS is required; explicitly select checked public model IDs")
    if SCORE_BUNDLE is None:
        raise ValueError("SCORE_BUNDLE is required; prepare and validate a local bundle first")
    if NORMALIZATION_RUN is None:
        raise ValueError("NORMALIZATION_RUN is required; complete notebook 01 first")
    bundle = Path(SCORE_BUNDLE)
    m2 = Path(NORMALIZATION_RUN)
    selected_ids = tuple(PGS_IDS)
    output = Path("/content/genome-evidence-work") / "m7-polygenic-score"

validated_bundle = validate_polygenic_score_bundle(bundle)
result = calculate_polygenic_scores(
    m2,
    bundle,
    output,
    ScoreConfig(pgs_ids=selected_ids, genotype_source_policy=GENOTYPE_SOURCE_POLICY),
)
manifest = json.loads((output / "manifest.json").read_text())
for relative, digest in manifest["artifacts"].items():
    assert file_sha256((output / relative).read_bytes()).hexdigest() == digest
completion = json.loads((output / "COMPLETED.json").read_text())
assert completion["run_id"] == result.run_id
assert json.loads((output / "polygenic_score_qc.json").read_text())["model_count"] == len(
    selected_ids
)
results = pl.read_parquet(output / "polygenic_score_results.parquet")
assert results.height == len(selected_ids)
assert pl.read_parquet(output / "score_contributions.parquet").height >= 1
assert "not a probability" in (output / "report.md").read_text()
if PROFILE == "synthetic_ci":
    row = results.to_dicts()[0]
    assert row["raw_partial_score"] == "1"
    assert row["status"] == "partial_not_evaluable"
    assert row["observed_contributed_count"] == 2
    assert row["imputed_contributed_count"] == 0
print(
    {
        "bundle_id": validated_bundle.bundle_id,
        "models": selected_ids,
        "statuses": result.statuses,
        "artifact_count": len(manifest["artifacts"]),
    }
)

## Next step
Personal mode requires a compatible completed M2 run and an explicitly selected, validated local score bundle. No network acquisition occurs during analysis.
